# 20 — Evaluate Text Model

Loads the best checkpoint for each trained model and runs full test-set
evaluation: classification report, confusion matrix, and inference speed.

**Depends on:** `01_preprocess_text.ipynb` (run automatically below).

**Reads from** `../experiments/text_unimodal/<version>/<model_slug>/model/`.

**Writes to each run directory:**
- `classification_report_test.json`
- `confusion_matrix_test.png`
- `eval_metrics.json` — full test metrics + inference speed

In [ ]:
# Imports train_df, dev_df, test_df, label2id, id2label, labels_sorted
%run 01_preprocess_text.ipynb

In [ ]:
import os
import json
import pandas as pd

from src.config import EXP_ROOT, VERSION, MODELS_TO_RUN, DEVICE
from src.training.utils import slugify, write_json
from src.evaluation.evaluate import run_full_evaluation

## Discover run directories

We look for the first (non-`__run*`) run directory for each model.
Change `run_dirs` manually if you want to evaluate a specific run.

In [ ]:
def find_run_dir(model_name: str) -> str:
    """Return the primary run directory for a model (no __run* suffix)."""
    slug = slugify(model_name)
    path = os.path.join(EXP_ROOT, VERSION, slug)
    assert os.path.isdir(path), f'Run directory not found: {path}\nRun 10_train_text_model.ipynb first.'
    return path

run_dirs = [find_run_dir(m) for m in MODELS_TO_RUN]
print('Evaluating:', run_dirs)

## Run evaluation

In [ ]:
all_results = {}

for model_name, run_dir in zip(MODELS_TO_RUN, run_dirs):
    print(f'\n=== {model_name} ===')
    results = run_full_evaluation(
        run_dir=run_dir,
        test_df=test_df,
        id2label=id2label,
        device=DEVICE,
    )
    all_results[model_name] = results
    write_json(os.path.join(run_dir, 'eval_metrics.json'), results)

    print(f'  Accuracy    : {results["accuracy"]:.4f}')
    print(f'  Macro F1    : {results["macro_f1"]:.4f}')
    print(f'  Weighted F1 : {results["weighted_f1"]:.4f}')
    print(f'  Infer ms/sample (bs=1): {results["inference_ms_per_sample_bs1"]:.2f}')

## Side-by-side summary

In [ ]:
summary = pd.DataFrame([
    {
        'model':        name,
        'accuracy':     r['accuracy'],
        'macro_f1':     r['macro_f1'],
        'weighted_f1':  r['weighted_f1'],
        'infer_ms_bs1': r['inference_ms_per_sample_bs1'],
    }
    for name, r in all_results.items()
])
summary

## Confusion matrices

Figures are already saved to each run directory. Display them here inline.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, len(run_dirs), figsize=(9 * len(run_dirs), 7), squeeze=False)

for ax, run_dir in zip(axes[0], run_dirs):
    img_path = os.path.join(run_dir, 'confusion_matrix_test.png')
    if os.path.exists(img_path):
        ax.imshow(mpimg.imread(img_path))
    ax.axis('off')

plt.tight_layout()
plt.show()